# ⚡ GPU-Accelerated Vector Search — Google Colab (T4 GPU)

**Before running:** Runtime → Change runtime type → **T4 GPU**

This notebook builds a production-grade semantic search service over 200K MS-MARCO passages using FAISS + NVIDIA cuVS.

| Step | Description |
|---|---|
| 0 | Mount Google Drive |
| 1 | Install dependencies |
| 2 | Verify GPU + NVML |
| 3 | Download data |
| 4 | Generate embeddings |
| 5 | Build CPU HNSW + GPU IVF-PQ + GPU CAGRA |
| 6 | Benchmark recall / latency / energy / QPS |
| 7 | nprobe sweep — recall vs latency Pareto |
| 8 | gRPC server demo |
| 9 | Artifact summary |

## Step 0 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, json
BASE_DIR = '/content/drive/MyDrive/ColabNotebooks/gpu-vector-search'
for d in ['data','embeddings','indexes','benchmarks','generated','proto']:
    os.makedirs(f'{BASE_DIR}/{d}', exist_ok=True)
print(f'Base dir: {BASE_DIR}')

Mounted at /content/drive
Base dir: /content/drive/MyDrive/ColabNotebooks/gpu-vector-search


## Step 1 — Install Dependencies

`faiss-gpu-cuvs` must be installed along with "..." packages since the Colab's pre-installed versions becomes mismatched. A runtime restart may be prompted after the conda cell.

In [2]:
!pip install -q \
    faiss-gpu-cuvs \
    "cudf-cu12==25.10.*" \
    "cuml-cu12==25.10.*" \
    "cugraph-cu12==25.10.*" \
    "raft-dask-cu12==25.10.*" \
    "cuvs-cu12==25.10.*" \
    "nx-cugraph-cu12==25.10.*" \
    "cudf-polars-cu12==25.10.*" \
    sentence-transformers==3.4.1 \
    grpcio==1.71.2 \
    grpcio-tools==1.71.2 \
    nvidia-ml-py==12.560.30 \
    tqdm==4.67.1 \
    datasets \
    jedi \
    --extra-index-url=https://pypi.nvidia.com

!pip check

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 65.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 176.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 252.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 211.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 253.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 149.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.2/224.2 kB 170.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.3/627.3 MB 22.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 33.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.9/913.9 kB 172.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.4/159.4 kB 118.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 2 — Verify GPU

In [3]:
import faiss, torch, pynvml, numpy as np

print(f'FAISS version  : {faiss.__version__}')
print(f'GPU count      : {faiss.get_num_gpus()}')
assert faiss.get_num_gpus() >= 1, 'No GPU — switch to T4 runtime!'
print(f'GPU name       : {torch.cuda.get_device_name(0)}')
props = torch.cuda.get_device_properties(0)
print(f'VRAM           : {props.total_memory/1024**3:.1f} GB')
print(f'Compute cap.   : {props.major}.{props.minor}')
assert props.major >= 7, f'CC {props.major}.{props.minor} not supported — need CC>=7.0'

pynvml.nvmlInit()
nvml_h = pynvml.nvmlDeviceGetHandleByIndex(0)
e0 = pynvml.nvmlDeviceGetTotalEnergyConsumption(nvml_h)
print(f'NVML energy API: OK ({e0:,} mJ cumulative)')

# Quick CAGRA smoke test
res_test = faiss.StandardGpuResources()
cfg_test  = faiss.GpuIndexCagraConfig()
ti = faiss.GpuIndexCagra(res_test, 64, faiss.METRIC_INNER_PRODUCT, cfg_test)
ti.train(np.random.randn(1000,64).astype('float32'))
print(f'CAGRA smoke test: OK  ntotal={ti.ntotal}')
print('\nAll checks passed.')

FAISS version  : 1.14.1
GPU count      : 1
GPU name       : Tesla T4
VRAM           : 14.6 GB
Compute cap.   : 7.5
NVML energy API: OK (15,472,266 mJ cumulative)
CAGRA smoke test: OK  ntotal=1000

All checks passed.


## Step 3 — Download Data

200,000 MS-MARCO passages + 500 dev queries. Skips if already on Drive.

In [5]:
from datasets import load_dataset
from tqdm import tqdm

DATA_DIR   = f'{BASE_DIR}/data'
pass_path  = f'{DATA_DIR}/passages_200k.jsonl'
query_path = f'{DATA_DIR}/dev_queries.jsonl'
MAX_PASS   = 200_000
MAX_Q      = 500

if os.path.exists(pass_path):
    print('[SKIP] passages_200k.jsonl already on Drive')
else:
    print(f'Streaming {MAX_PASS:,} MS-MARCO passages...')
    dataset = load_dataset('ms_marco','v1.1',split='train',streaming=True)  # ,trust_remote_code=True)
    passages, pids, count = [], [], 0
    for ex in tqdm(dataset, total=MAX_PASS):
        for text in ex['passages']['passage_text']:
            passages.append(text)
            pids.append(f"{ex['query_id']}_{count}")
            count += 1
            if count >= MAX_PASS: break
        if count >= MAX_PASS: break
    with open(pass_path,'w') as f:
        for pid,text in zip(pids[:MAX_PASS], passages[:MAX_PASS]):
            f.write(json.dumps({'id':pid,'text':text})+'\n')
    print(f'  {MAX_PASS:,} passages saved')

if os.path.exists(query_path):
    print('[SKIP] dev_queries.jsonl already on Drive')
else:
    dev = load_dataset('ms_marco','v1.1',split='validation')  # ,trust_remote_code=True)
    with open(query_path,'w') as f:
        for ex in list(dev)[:MAX_Q]:
            f.write(json.dumps({'id':str(ex['query_id']),'text':ex['query']})+'\n')
    print(f'  {MAX_Q} queries saved')

Streaming 200,000 MS-MARCO passages...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

 12%|█▏        | 24345/200000 [00:15<01:49, 1598.68it/s]


  200,000 passages saved


...

v1.1/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10047 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/82326 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9650 [00:00<?, ? examples/s]

  500 queries saved


## Step 4 — Generate Embeddings

Encodes with `all-MiniLM-L6-v2` (384-dim). L2-normalised so inner product = cosine similarity.

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMB_DIR  = f'{BASE_DIR}/embeddings'
BATCH    = 512
p_emb_f  = f'{EMB_DIR}/passage_embs.npy'
q_emb_f  = f'{EMB_DIR}/query_embs.npy'

model = SentenceTransformer('all-MiniLM-L6-v2')
DIM   = model.get_sentence_embedding_dimension()
print(f'Model: all-MiniLM-L6-v2  dim={DIM}')

if os.path.exists(p_emb_f):
    print('[SKIP] passage embeddings found on Drive')
    passage_embs = np.load(p_emb_f)
else:
    texts = [json.loads(l)['text'] for l in open(pass_path)]
    passage_embs = model.encode(texts, batch_size=BATCH, show_progress_bar=True,
                                convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    np.save(p_emb_f, passage_embs)
    print(f'  passage_embs: {passage_embs.shape}')

if os.path.exists(q_emb_f):
    print('[SKIP] query embeddings found on Drive')
    query_embs = np.load(q_emb_f)
else:
    qtexts = [json.loads(l)['text'] for l in open(query_path)]
    query_embs = model.encode(qtexts, batch_size=BATCH, show_progress_bar=True,
                               convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    np.save(q_emb_f, query_embs)
    print(f'  query_embs: {query_embs.shape}')

N, D = passage_embs.shape
print(f'\nCorpus: {N:,} vectors x dim={D}  ({N*D*4/1024**2:.0f} MB raw)  T4 VRAM=16384 MB OK')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Model: all-MiniLM-L6-v2  dim=384


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

  passage_embs: (200000, 384)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  query_embs: (500, 384)

Corpus: 200,000 vectors x dim=384  (293 MB raw)  T4 VRAM=16384 MB OK


## Step 5 — Build Indexes

### Three indexes, three strategies

| Index | Algorithm | Why |
|---|---|---|
| CPU HNSW | Graph-based ANN on CPU | Baseline — zero GPU cost |
| GPU IVF-PQ | Inverted file + PQ compression via cuVS | 48x smaller, very fast build |
| GPU CAGRA | GPU-native graph ANN via cuVS | Highest recall and throughput |

### Energy Measurement
`nvmlDeviceGetTotalEnergyConsumption` (Volta+) returns cumulative mJ since driver load. Delta start→stop = interval energy. `torch.cuda.synchronize()` ensures GPU work finishes before reading the counter.

In [7]:
import time, faiss

IDX_DIR   = f'{BASE_DIR}/indexes'
BENCH_DIR = f'{BASE_DIR}/benchmarks'
os.makedirs(IDX_DIR, exist_ok=True)
os.makedirs(BENCH_DIR, exist_ok=True)

def energy_mj(): return pynvml.nvmlDeviceGetTotalEnergyConsumption(nvml_h)

def timed_energy(fn):
    torch.cuda.synchronize()
    e0, t0 = energy_mj(), time.perf_counter()
    result = fn()
    torch.cuda.synchronize()
    return result, time.perf_counter()-t0, (energy_mj()-e0)/1000.0

res  = faiss.StandardGpuResources()
build_stats = {}

# ===== A: CPU HNSW (baseline) =====
# M=32 edges/node; efConstruction=200 for high-quality graph
hnsw_path = f'{IDX_DIR}/hnsw_cpu.faiss'
if os.path.exists(hnsw_path):
    print('[SKIP] hnsw_cpu.faiss'); hnsw = faiss.read_index(hnsw_path); hnsw.hnsw.efSearch=128
else:
    print('Building CPU HNSW (M=32, efConstruction=200)...')
    t0=time.perf_counter()
    hnsw = faiss.IndexHNSWFlat(D, 32, faiss.METRIC_INNER_PRODUCT)
    hnsw.hnsw.efConstruction = 200
    hnsw.add(passage_embs)
    elapsed = time.perf_counter()-t0
    hnsw.hnsw.efSearch = 128
    faiss.write_index(hnsw, hnsw_path)
    build_stats['hnsw_cpu'] = {'build_s':round(elapsed,2),'ntotal':hnsw.ntotal}
    print(f'  HNSW done: {elapsed:.1f}s  ntotal={hnsw.ntotal:,}')

# ===== B: GPU IVF-PQ (cuVS) =====
# nlist=512 (~sqrt(200K)), M=32 sub-vectors -> 32 bytes/vector (48x compression)
ivfpq_path = f'{IDX_DIR}/ivfpq_gpu.faiss'
if os.path.exists(ivfpq_path):
    print('[SKIP] ivfpq_gpu.faiss')
    co_g=faiss.GpuClonerOptions(); co_g.use_cuvs=True
    ivfpq_gpu = faiss.index_cpu_to_gpu(res, 0, faiss.read_index(ivfpq_path), co_g)
    ivfpq_gpu.nprobe=16
else:
    print('Building GPU IVF-PQ (nlist=512, M=32, nbits=8) via cuVS...')
    cfg_p = faiss.GpuIndexIVFPQConfig(); cfg_p.use_cuvs = True
    def _ivfpq():
        idx = faiss.GpuIndexIVFPQ(res, D, 512, 32, 8, faiss.METRIC_INNER_PRODUCT, cfg_p)
        print(f'  Training on {min(N,20480):,} vectors...')
        idx.train(passage_embs[:min(N,20480)])
        idx.add(passage_embs)
        return idx
    ivfpq_gpu, elapsed, ej = timed_energy(_ivfpq)
    ivfpq_gpu.nprobe = 16
    faiss.write_index(faiss.index_gpu_to_cpu(ivfpq_gpu), ivfpq_path)
    build_stats['ivfpq_gpu']={'build_s':round(elapsed,2),'energy_j':round(ej,2),'ntotal':ivfpq_gpu.ntotal}
    print(f'  IVF-PQ done: {elapsed:.1f}s  energy={ej:.1f}J  ntotal={ivfpq_gpu.ntotal:,}')

# ===== C: GPU CAGRA (cuVS) =====
# GPU-native graph index. train() indexes ALL data — there is no separate add() step.
# intermediate_graph_degree=64: pre-pruning kNN neighbours
# graph_degree=32: final edges after pruning
# VRAM peak during build: N x (D*2 + 276) = ~200 MB for this config
cagra_path = f'{IDX_DIR}/cagra_gpu.faiss'
if os.path.exists(cagra_path):
    print('[SKIP] cagra_gpu.faiss')
    co_g2=faiss.GpuClonerOptions(); co_g2.use_cuvs=True
    cagra_gpu = faiss.index_cpu_to_gpu(res, 0, faiss.read_index(cagra_path), co_g2)
else:
    print('Building GPU CAGRA (graph_degree=32, intermediate=64) via cuVS...')
    cfg_c = faiss.GpuIndexCagraConfig()
    cfg_c.graph_degree = 32
    cfg_c.intermediate_graph_degree = 64
    cfg_c.build_algo = faiss.graph_build_algo_IVF_PQ
    def _cagra():
        idx = faiss.GpuIndexCagra(res, D, faiss.METRIC_INNER_PRODUCT, cfg_c)
        idx.train(passage_embs)  # CAGRA: train() indexes everything
        return idx
    cagra_gpu, elapsed, ej = timed_energy(_cagra)
    faiss.write_index(faiss.index_gpu_to_cpu(cagra_gpu), cagra_path)
    # Bonus: export GPU-built CAGRA graph as CPU HNSW
    # This is a unique cuVS feature: build a high-quality HNSW on GPU
    # and export a standard CPU HNSW that needs no GPU at serving time.
    hc = faiss.IndexHNSWCagra(); hc.base_level_only = False
    cagra_gpu.copyTo(hc)
    faiss.write_index(hc, f'{IDX_DIR}/hnsw_from_cagra.faiss')
    build_stats['cagra_gpu']={'build_s':round(elapsed,2),'energy_j':round(ej,2),'ntotal':cagra_gpu.ntotal}
    print(f'  CAGRA done: {elapsed:.1f}s  energy={ej:.1f}J  ntotal={cagra_gpu.ntotal:,}')
    print(f'  CAGRA->HNSW export saved')

with open(f'{BENCH_DIR}/build_stats.json','w') as f: json.dump(build_stats,f,indent=2)
print('Build stats saved.')

Building CPU HNSW (M=32, efConstruction=200)...
  HNSW done: 194.4s  ntotal=200,000
Building GPU IVF-PQ (nlist=512, M=32, nbits=8) via cuVS...
  Training on 20,480 vectors...
  IVF-PQ done: 0.8s  energy=32.7J  ntotal=200,000
Building GPU CAGRA (graph_degree=32, intermediate=64) via cuVS...
  CAGRA done: 33.2s  energy=1901.4J  ntotal=200,000
  CAGRA->HNSW export saved
Build stats saved.


## Step 6 — Benchmark: Recall, Latency & Energy

For each index:
- **Recall@10** vs brute-force ground truth
- **P50/P95/P99 single-query latency** (ms)
- **Energy per 1000 queries** (Joules)
- **Batch QPS** at multiple batch sizes

`torch.cuda.synchronize()` is called before energy counter reads to ensure GPU work is complete.

In [8]:
import numpy as np, time, json

K = 10
BATCH_SIZES = [1,10,50,100,500,1000]

# Ground truth: exact brute-force on GPU
print('Computing brute-force ground truth...')
flat_gpu = faiss.GpuIndexFlatIP(res, D)
flat_gpu.add(passage_embs)
torch.cuda.synchronize()
_, GT = flat_gpu.search(query_embs, K)
print(f'  GT shape: {GT.shape}')

co_r = faiss.GpuClonerOptions(); co_r.use_cuvs = True

def load_idx(fname, nprobe=None):
    idx_cpu = faiss.read_index(f'{IDX_DIR}/{fname}')
    if 'hnsw_cpu' in fname: idx_cpu.hnsw.efSearch=128; return idx_cpu
    g = faiss.index_cpu_to_gpu(res, 0, idx_cpu, co_r)
    if nprobe: g.nprobe = nprobe
    return g

indexes = {
    'CPU HNSW':          load_idx('hnsw_cpu.faiss'),
    'GPU IVF-PQ (cuVS)': load_idx('ivfpq_gpu.faiss', 16),
    'GPU CAGRA (cuVS)':  load_idx('cagra_gpu.faiss'),
}

def sync_gpu(idx):
    if hasattr(idx,'getDevice'): torch.cuda.synchronize()

def recall_at_k(pred, gt, k):
    return sum(len(set(p.tolist())&set(g.tolist())) for p,g in zip(pred,gt))/(len(gt)*k)

def latency_profile(idx, queries, k, warmup=50, trials=500):
    lats=[]
    for i in range(warmup+trials):
        q=queries[i%len(queries)].reshape(1,-1)
        sync_gpu(idx); t0=time.perf_counter()
        idx.search(q,k)
        sync_gpu(idx)
        if i>=warmup: lats.append((time.perf_counter()-t0)*1000)
    a=np.array(lats)
    return {'p50_ms':round(float(np.percentile(a,50)),3),
            'p95_ms':round(float(np.percentile(a,95)),3),
            'p99_ms':round(float(np.percentile(a,99)),3)}

def energy_1kq(idx, queries, k):
    def run():
        for i in range(1000): idx.search(queries[i%len(queries)].reshape(1,-1),k)
    _,dur,ej = timed_energy(run)
    return round(ej,2), round(dur,2)

all_results={}
print(f'\n{"Index":<22} {"Recall@10":>10} {"P50ms":>7} {"P95ms":>7} {"P99ms":>7} {"E/1kQ J":>10}')
print('-'*68)

for name,idx in indexes.items():
    _,pred = idx.search(query_embs,K)
    r10  = recall_at_k(pred,GT,K)
    lat  = latency_profile(idx,query_embs,K)
    ej,_ = energy_1kq(idx,query_embs,K)
    qps  = {}
    for bs in BATCH_SIZES:
        batch=query_embs[:bs]
        for _ in range(3): idx.search(batch,K)
        sync_gpu(idx); t0=time.perf_counter()
        for _ in range(20): idx.search(batch,K)
        sync_gpu(idx)
        qps[str(bs)]=round((20*bs)/(time.perf_counter()-t0),1)
    print(f'{name:<22} {r10*100:>9.1f}% {lat["p50_ms"]:>7} {lat["p95_ms"]:>7} {lat["p99_ms"]:>7} {ej:>10}')
    all_results[name]={'recall_at_10':round(r10,4),'latency':lat,
                       'energy_1000q_joules':ej,'qps_by_batch':qps}

with open(f'{BENCH_DIR}/results.json','w') as f: json.dump(all_results,f,indent=2)
print('\nResults saved.')

Computing brute-force ground truth...
  GT shape: (500, 10)

Index                   Recall@10   P50ms   P95ms   P99ms    E/1kQ J
--------------------------------------------------------------------
CPU HNSW                    99.1%   0.808   1.186   1.369       27.5
GPU IVF-PQ (cuVS)           53.1%   0.327   0.404    0.55      11.04
GPU CAGRA (cuVS)            94.2%   0.447   0.547   0.612      20.43

Results saved.


## Step 7 — nprobe Sweep

nprobe = number of Voronoi cells searched (out of 512 total). This sweep produces the recall-vs-latency Pareto curve for IVF-PQ.

In [9]:
NPROBE_VALS = [1,4,8,16,32,64,128,256]
ivfpq_idx = indexes['GPU IVF-PQ (cuVS)']
print(f'{"nprobe":>8}  {"Recall@10":>12}  {"P50 ms":>10}  {"E/1kQ J":>12}')
print('-'*50)
sweep=[]
for np_v in NPROBE_VALS:
    ivfpq_idx.nprobe=np_v
    _,pred=ivfpq_idx.search(query_embs,K)
    r=recall_at_k(pred,GT,K)
    lat_n=latency_profile(ivfpq_idx,query_embs,K,warmup=10,trials=100)
    ej,_=energy_1kq(ivfpq_idx,query_embs,K)
    print(f'{np_v:>8}  {r*100:>11.1f}%  {lat_n["p50_ms"]:>9.2f}ms  {ej:>11.2f}J')
    sweep.append({'nprobe':np_v,'recall':round(r,4),'p50_ms':lat_n['p50_ms'],'energy_1kq_j':ej})
all_results['nprobe_sweep']=sweep
with open(f'{BENCH_DIR}/results.json','w') as f: json.dump(all_results,f,indent=2)
print('nprobe sweep saved.')

  nprobe     Recall@10      P50 ms       E/1kQ J
--------------------------------------------------
       1         32.4%       0.32ms        10.77J
       4         47.5%       0.34ms        10.87J
       8         50.9%       0.32ms        11.02J
      16         53.1%       0.33ms        11.09J
      32         54.2%       0.36ms        11.36J
      64         54.7%       0.41ms        11.75J
     128         54.9%       0.40ms        19.82J
     256         54.9%       0.53ms        26.47J
nprobe sweep saved.


## Step 8 — gRPC Server Demo

Compile protobuf stubs, start async gRPC server in a background thread, then run a client demo.

In [10]:
import sys, threading, asyncio, grpc, time

GEN_DIR   = f'{BASE_DIR}/generated'
PROTO_DIR = f'{BASE_DIR}/proto'
sys.path.insert(0, GEN_DIR)

PROTO = '''syntax="proto3"; package vectorsearch;
service SearchService { rpc Search(SearchRequest) returns(SearchResponse); rpc HealthCheck(HealthRequest) returns(HealthResponse); }
message SearchRequest  { string query_text=1; int32 top_k=2; string index_type=3; }
message SearchResult   { string passage_id=1; string passage_text=2; float score=3; int32 rank=4; }
message SearchResponse { repeated SearchResult results=1; float latency_ms=2; string index_used=3; }
message HealthRequest  {}
message HealthResponse { string status=1; int32 vectors_indexed=2; string gpu_name=3; }'''

with open(f'{PROTO_DIR}/search.proto','w') as f: f.write(PROTO)
!python -m grpc_tools.protoc -I{PROTO_DIR} --python_out={GEN_DIR} --grpc_python_out={GEN_DIR} {PROTO_DIR}/search.proto
import search_pb2, search_pb2_grpc
print('Proto stubs compiled.')

Proto stubs compiled.


In [11]:
from sentence_transformers import SentenceTransformer as STM

_enc  = STM('all-MiniLM-L6-v2')
_pids = []
_ptxt = {}
with open(pass_path) as f:
    for line in f:
        obj=json.loads(line); _ptxt[obj['id']]=obj['text']; _pids.append(obj['id'])

_idxs = {
    'hnsw':  load_idx('hnsw_cpu.faiss'),
    'ivfpq': load_idx('ivfpq_gpu.faiss', 16),
    'cagra': load_idx('cagra_gpu.faiss'),
}

class _Svc(search_pb2_grpc.SearchServiceServicer):
    async def Search(self,req,ctx):
        k=max(req.top_k,1); key=req.index_type if req.index_type in _idxs else 'cagra'
        q=_enc.encode([req.query_text],normalize_embeddings=True,convert_to_numpy=True).astype('float32')
        t0=time.perf_counter(); D_,I_=_idxs[key].search(q,k); ms=(time.perf_counter()-t0)*1000
        rl=[search_pb2.SearchResult(passage_id=_pids[int(i)],
                                    passage_text=_ptxt.get(_pids[int(i)],'')[:400],
                                    score=float(d),rank=rk)
            for rk,(d,i) in enumerate(zip(D_[0],I_[0]),1) if i>=0]
        return search_pb2.SearchResponse(results=rl,latency_ms=ms,index_used=key)
    async def HealthCheck(self,req,ctx):
        return search_pb2.HealthResponse(status='ok',
                                         vectors_indexed=_idxs['cagra'].ntotal,
                                         gpu_name=torch.cuda.get_device_name(0))

async def _srv():
    s=grpc.aio.server()
    search_pb2_grpc.add_SearchServiceServicer_to_server(_Svc(),s)
    s.add_insecure_port('[::]:50051'); await s.start(); await s.wait_for_termination()

def _run(): loop=asyncio.new_event_loop(); asyncio.set_event_loop(loop); loop.run_until_complete(_srv())
threading.Thread(target=_run,daemon=True).start()
time.sleep(3); print('gRPC server running on :50051')

gRPC server running on :50051


In [12]:
import logging
logging.getLogger('asyncio').setLevel(logging.CRITICAL)

async def demo():
    async with grpc.aio.insecure_channel('localhost:50051') as ch:
        stub = search_pb2_grpc.SearchServiceStub(ch)
        h = await stub.HealthCheck(search_pb2.HealthRequest())
        print(f'Health: {h.status}  GPU: {h.gpu_name}  Vectors: {h.vectors_indexed:,}')
        for q in ['What is machine learning?','How does attention work in transformers?']:
            print(f'\nQuery: {q}')
            for it in ['hnsw','ivfpq','cagra']:
                r = await stub.Search(search_pb2.SearchRequest(query_text=q,top_k=3,index_type=it))
                top = r.results[0].passage_text[:80] if r.results else 'N/A'
                print(f'  [{it:<5}] {r.latency_ms:.2f}ms  -> {top}...')
await demo()

Health: ok  GPU: Tesla T4  Vectors: 200,000

Query: What is machine learning?
  [hnsw ] 1.79ms  -> Machine learning is grounded in hard sciences, thus it is one of the more rare b...
  [ivfpq] 2.91ms  -> Used by over 60 million people worldwide, Lumosity creates a Training Program th...
  [cagra] 1.11ms  -> Machine learning is grounded in hard sciences, thus it is one of the more rare b...

Query: How does attention work in transformers?
  [hnsw ] 1.32ms  -> For example, the eye receives visual information and codes information into elec...
  [ivfpq] 0.45ms  -> Attention span is the amount of concentrated time one can spend on a task withou...
  [cagra] 1.10ms  -> For example, the eye receives visual information and codes information into elec...


## Step 9 — Summary & Cleanup

In [13]:
print('=== ARTIFACTS ON GOOGLE DRIVE ===')
for sub in ['data','embeddings','indexes','benchmarks']:
    path=f'{BASE_DIR}/{sub}'
    if os.path.exists(path):
        files=os.listdir(path)
        total_mb=sum(os.path.getsize(f'{path}/{ff}') for ff in files if os.path.isfile(f'{path}/{ff}'))/1024**2
        print(f'  {sub}/  ({len(files)} files, {total_mb:.0f} MB)')
        for ff in sorted(files):
            sz=os.path.getsize(f'{path}/{ff}')/1024**2
            print(f'    {ff}  ({sz:.1f} MB)')
print('\n=== BENCHMARK SUMMARY ===')
with open(f'{BENCH_DIR}/results.json') as f: data=json.load(f)
for name,v in data.items():
    if name=='nprobe_sweep': continue
    print(f'\n{name}')
    print(f'  Recall@10          : {v["recall_at_10"]*100:.1f}%')
    print(f'  P50 latency        : {v["latency"]["p50_ms"]} ms')
    print(f'  Energy / 1k queries: {v["energy_1000q_joules"]} J')
pynvml.nvmlShutdown()
print('\nDone. All artifacts saved to Google Drive.')

=== ARTIFACTS ON GOOGLE DRIVE ===
  data/  (2 files, 88 MB)
    dev_queries.jsonl  (0.0 MB)
    passages_200k.jsonl  (87.5 MB)
  embeddings/  (2 files, 294 MB)
    passage_embs.npy  (293.0 MB)
    query_embs.npy  (0.7 MB)
  indexes/  (4 files, 995 MB)
    cagra_gpu.faiss  (320.5 MB)
    hnsw_cpu.faiss  (344.9 MB)
    hnsw_from_cagra.faiss  (320.5 MB)
    ivfpq_gpu.faiss  (8.8 MB)
  benchmarks/  (1 files, 0 MB)
    results.json  (0.0 MB)

=== BENCHMARK SUMMARY ===

CPU HNSW
  Recall@10          : 99.1%
  P50 latency        : 0.808 ms
  Energy / 1k queries: 27.5 J

GPU IVF-PQ (cuVS)
  Recall@10          : 53.1%
  P50 latency        : 0.327 ms
  Energy / 1k queries: 11.04 J

GPU CAGRA (cuVS)
  Recall@10          : 94.2%
  P50 latency        : 0.447 ms
  Energy / 1k queries: 20.43 J

Done. All artifacts saved to Google Drive.
